# Sampling Methods

In [5]:
import numpy as np

## Uniform vs. Stratified Sampling 

**Uniform sampling** picks points with equal probability across the entire space. While simple, it often leaves "gaps" or clusters data points awkwardly.

**Stratified sampling** ensures that sub-groups (strata) are represented. In a 1D space $[0, 1]$, this means dividing the range into $N$ equal bins and picking one point per bin. This reduces variance and ensures better coverage of the space.

In [6]:
def sample_1d(n_samples, method="uniform"):
    if method == "uniform":
        # Every point has an equal 1/N chance
        return np.random.uniform(0, 1, n_samples)
    
    elif method == "stratified":
        # Divide [0, 1] into n_samples bins
        bins = np.linspace(0, 1, n_samples + 1)
        low = bins[:-1]
        high = bins[1:]
        # Sample one point within each bin
        return np.random.uniform(low, high)



## Reservoir Sampling 

This is a classic "streaming" algorithm. Use this when you have a dataset so large it doesn't fit in memory, or an incoming stream of unknown size, and you need to maintain a representative sample of size $k$.

The Logic:
1. Fill the "reservoir" with the first $k$ elements.
2. For every subsequent element $i$ (where $i > k$), pick a random integer $j$ between $0$ and $i$.
3. If $j < k$, replace `reservoir[j]` with the new element.

In [7]:
def reservoir_sampling(stream, k):
    # Initialize reservoir with first k elements
    reservoir = np.copy(stream[:k])
    
    for i in range(k, len(stream)):
        # Generate a random index between 0 and i (inclusive)
        j = np.random.randint(0, i + 1)
        
        # If the index falls within the reservoir range, replace it
        if j < k:
            reservoir[j] = stream[i]
            
    return reservoir


## Generating Data from Distributions

In an interview, you might be asked to transform a $Uniform(0, 1)$ distribution into something else.

### Inverse Transform Sampling

If you have a probability density function (PDF) $f(x)$ and its cumulative distribution function (CDF) $F(x)$, you can generate samples by:
- Generating $u \sim Uniform(0, 1)$.
- Computing $x = F^{-1}(u)$.
 
For an Exponential Distribution with rate $\lambda$:

The CDF is $F(x) = 1 - e^{-\lambda x}$.
Solving for $x$ gives the inverse:

$$x = -\frac{\ln(1-u)}{\lambda}$$


In [8]:

def sample_exponential(n_samples, lambd=1.0):
    u = np.random.uniform(0, 1, n_samples)
    # Since 1-u is also Uniform(0,1), we can just use u
    return -np.log(1 - u) / lambd


The **Box-Muller Transform** (Generating Normals)

To generate $Normal(0, 1)$ samples from $Uniform(0, 1)$ without using `np.random.normal`:

$$Z_0 = \sqrt{-2\ln U_1} \cos(2\pi U_2)$$

$$Z_1 = \sqrt{-2\ln U_1} \sin(2\pi U_2)$$

In [9]:

def sample_normal_box_muller(n_samples):
    # n_samples should be even for this implementation
    u1 = np.random.uniform(0, 1, n_samples // 2)
    u2 = np.random.uniform(0, 1, n_samples // 2)
    
    r = np.sqrt(-2 * np.log(u1))
    theta = 2 * np.pi * u2
    
    z0 = r * np.cos(theta)
    z1 = r * np.sin(theta)
    
    return np.concatenate([z0, z1])


With Imbalanced Data (e.g., millions of miles of straight driving vs. a few seconds of a pedestrian jumping into the road):
- Stratified Sampling is used to ensure those rare "edge cases" are included in training batches.
- Reservoir Sampling is used to sample from huge logs of LIDAR data streaming off a vehicle.